# 01 — Factor Momentum Foundations

> Esta investigación pregunta si parte del momentum observado en activos puede entenderse como exposición a factores o grupos sistemáticos que también presentan persistencia temporal.

$ R_{i,t} = \sum_{k=1}^{K} \beta_{ik}F_{k,t} + \epsilon_{i,t} $

La primera etapa usará proxies ETF observables y líquidos, no factores académicos puros. La ecuación es un marco de investigación; no supone todavía que los proxies expliquen los retornos de los activos.


## <b>Relación con TSMOM</b>

`01_tsmom` ya establece un baseline de momentum temporal. Allí la pregunta es: ¿el activo viene subiendo o bajando respecto a su propia historia? Aquí la pregunta cambia a: ¿ese momentum coincide con momentum persistente en factores o grupos sistemáticos?

Esta línea complementa el baseline; no lo reemplaza ni reutiliza todavía abstracciones de código comunes.


## <b>Datos</b>

> Todo análisis posterior leerá exclusivamente `data/gold/daily_series/etf/<SYMBOL>.parquet`. Raw, Silver, `series/` e intradía quedan fuera del análisis.

El universo proxy inicial propuesto es SPY, QQQ, IWM, IWD, IWF, XLE y XLF. La disponibilidad se comprobará antes de calcular retornos.


## <b>Block 01 — Factor universe</b>

> Verificar el universo de proxies ETF disponible en Gold, su cobertura común y los límites de interpretación de cada proxy.

Estos ETFs son proxies económicos observables. En esta primera etapa permiten estudiar persistencia relativa entre estilos y sectores sin asumir todavía una estructura factorial académica exacta.


In [ ]:
from datetime import date
from pathlib import Path

import plotly.graph_objects as go
import polars as pl

PROXIES = {
    "SPY": "broad US equity market",
    "QQQ": "large-cap growth / technology tilt",
    "IWM": "small-cap tilt",
    "IWD": "value tilt",
    "IWF": "growth tilt",
    "XLE": "energy sector",
    "XLF": "financial sector",
}
RESEARCH_START = date(2018, 1, 2)
RESEARCH_END = date(2026, 11, 17)

def find_project_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        for project_root in (candidate, candidate / "Momentum_systematic_strategies"):
            if (project_root / "data" / "gold" / "daily_series" / "etf").is_dir():
                return project_root
    raise FileNotFoundError("No se encontró data/gold/daily_series/etf desde el directorio actual.")

PROJECT_ROOT = find_project_root(Path.cwd())
GOLD_ROOT = PROJECT_ROOT / "data" / "gold" / "daily_series" / "etf"


In [ ]:
frames = {}
audit_rows = []

for symbol, proxy_role in PROXIES.items():
    path = GOLD_ROOT / f"{symbol}.parquet"
    if not path.exists():
        raise FileNotFoundError(f"Gold no disponible para {symbol}: {path}")

    frame = pl.read_parquet(path).with_columns(pl.col("session_date").cast(pl.Date))
    if "adjusted_close" not in frame.columns:
        raise ValueError(f"{symbol} no contiene adjusted_close.")

    frame = frame.filter(
        pl.col("session_date").is_between(RESEARCH_START, RESEARCH_END, closed="both")
    ).sort("session_date")
    dates = frame.get_column("session_date").to_list()
    relevant_nulls = int(frame.select(["session_date", "adjusted_close"]).null_count().sum_horizontal()[0])
    duplicates = int(frame.get_column("session_date").is_duplicated().sum())

    if not frame.height or relevant_nulls or duplicates or dates != sorted(dates):
        raise ValueError(f"{symbol} no supera los controles básicos de cobertura.")
    if bool((frame.get_column("adjusted_close") <= 0).any()):
        raise ValueError(f"{symbol} contiene adjusted_close no positivo.")

    frames[symbol] = frame.select("session_date", "adjusted_close")
    audit_rows.append({
        "symbol": symbol,
        "proxy_role": proxy_role,
        "start_date": frame.get_column("session_date").min(),
        "end_date": frame.get_column("session_date").max(),
        "n_sessions": frame.height,
        "nulls": relevant_nulls,
        "duplicates": duplicates,
    })

universe_audit = pl.DataFrame(audit_rows)
universe_audit


La tabla confirma `adjusted_close`, precios positivos, fechas ordenadas, ausencia de duplicados y nulos relevantes. El control de calidad detallado ya corresponde a la promoción Gold.


In [ ]:
common_panel = frames["SPY"].rename({"session_date": "date", "adjusted_close": "SPY"})
for symbol in list(PROXIES)[1:]:
    common_panel = common_panel.join(
        frames[symbol].rename({"session_date": "date", "adjusted_close": symbol}),
        on="date",
        how="inner",
    )

common_panel = common_panel.sort("date")
panel_summary = {
    "dimensions": common_panel.shape,
    "first_date": common_panel.get_column("date").min(),
    "last_date": common_panel.get_column("date").max(),
    "nulls": int(common_panel.null_count().sum_horizontal()[0]),
    "common_sessions": common_panel.height,
}
panel_summary


In [ ]:
normalized_prices = common_panel.select(
    "date",
    *[(pl.col(symbol) / pl.col(symbol).first()).alias(symbol) for symbol in PROXIES],
)

figure = go.Figure()
for symbol in PROXIES:
    figure.add_scatter(x=normalized_prices["date"], y=normalized_prices[symbol], mode="lines", name=symbol)

figure.update_layout(
    title="Factor proxy universe — normalized adjusted prices",
    yaxis_title="P_t / P_0",
    legend_title_text="",
)
figure.show()


### <b>Block 01 conclusion</b>

- Los siete proxies propuestos — SPY, QQQ, IWM, IWD, IWF, XLE y XLF — están disponibles en Gold.
- La ventana común exacta es `2018-01-02 → 2026-11-17`, con 2,232 sesiones.
- Cada serie conserva 2,232 observaciones, `adjusted_close` disponible, precios positivos y cero nulos o fechas duplicadas en la ventana.
- No quedan diferencias de cobertura entre los proxies dentro del rango de investigación.
- El panel `date | SPY | QQQ | IWM | IWD | IWF | XLE | XLF` queda preparado para construir factor returns.

El siguiente bloque estudiará los retornos diarios de estos proxies.


## <b>Block 02 — Factor returns</b>

> Los retornos se calculan sobre el panel Gold validado de Block 01. Son descripciones de proxies ETF, no retornos de factores académicos puros.

$ r_t = \ln(P_t / P_{t-1}) $


In [ ]:
return_columns = [f"{symbol}_ret" for symbol in PROXIES]
returns_panel = common_panel.with_columns([
    pl.col(symbol).log().diff().alias(f"{symbol}_ret") for symbol in PROXIES
]).drop_nulls(return_columns)

returns_panel.select("date", *return_columns).head()


In [ ]:
return_rows = []
for symbol in PROXIES:
    series = returns_panel.get_column(f"{symbol}_ret")
    return_rows.append({
        "symbol": symbol,
        "mean daily return (%)": series.mean() * 100,
        "annualized volatility (%)": series.std() * (252 ** 0.5) * 100,
        "min daily return (%)": series.min() * 100,
        "max daily return (%)": series.max() * 100,
        "valid observations": len(series),
    })

return_summary = pl.DataFrame(return_rows).with_columns(
    pl.col([
        "mean daily return (%)", "annualized volatility (%)",
        "min daily return (%)", "max daily return (%)",
    ]).round(3)
)
return_summary


### <b>Correlation structure</b>

Las correlaciones describen co-movimiento contemporáneo; no establecen causalidad ni sustituyen una exposición factorial estimada.


In [ ]:
correlation_table = returns_panel.select([
    pl.col(f"{symbol}_ret").alias(symbol) for symbol in PROXIES
]).corr().with_columns(pl.Series("proxy", list(PROXIES))).select("proxy", *PROXIES).with_columns(
    pl.all().exclude("proxy").round(3)
)

correlation_table


In [ ]:
figure = go.Figure(go.Heatmap(
    z=correlation_table.select(list(PROXIES)).to_numpy(),
    x=list(PROXIES),
    y=list(PROXIES),
    zmin=-1,
    zmax=1,
    colorscale="RdBu",
    zmid=0,
    texttemplate="%{z:.2f}",
))
figure.update_layout(title="Daily return correlation — factor proxy universe")
figure.show()


### <b>Relative return spreads</b>

> Los spreads son diferencias de retornos logarítmicos diarios entre proxies observables. Se usan para describir liderazgo relativo, no como factores académicos.


In [ ]:
SPREADS = {
    "IWF - IWD": ("IWF", "IWD"),
    "IWM - SPY": ("IWM", "SPY"),
    "QQQ - SPY": ("QQQ", "SPY"),
    "XLE - SPY": ("XLE", "SPY"),
    "XLF - SPY": ("XLF", "SPY"),
}

spread_columns = {label: f"spread_{long.lower()}_{short.lower()}" for label, (long, short) in SPREADS.items()}
spread_panel = returns_panel.with_columns([
    (pl.col(f"{long}_ret") - pl.col(f"{short}_ret")).alias(spread_columns[label])
    for label, (long, short) in SPREADS.items()
])

relative_performance = spread_panel.select(
    "date",
    *[(pl.col(column).cum_sum().exp()).alias(label) for label, column in spread_columns.items()],
)
relative_summary = pl.DataFrame([
    {
        "spread": label,
        "cumulative relative return (%)": (relative_performance.get_column(label)[-1] - 1) * 100,
    }
    for label in SPREADS
]).with_columns(pl.col("cumulative relative return (%)").round(3))
relative_summary


In [ ]:
figure = go.Figure()
for label in ("IWF - IWD", "IWM - SPY", "QQQ - SPY"):
    figure.add_scatter(x=relative_performance["date"], y=relative_performance[label], mode="lines", name=label)

figure.update_layout(
    title="Cumulative relative performance — selected proxy spreads",
    yaxis_title="relative wealth",
    legend_title_text="",
)
figure.show()


### <b>Rolling correlation and cross-sectional dispersion</b>

Las correlaciones móviles de 63 sesiones y la dispersión diaria muestran que el co-movimiento entre proxies puede cambiar con el tiempo.


In [ ]:
rolling_correlations = returns_panel.select(
    "date",
    *[
        pl.rolling_corr(pl.col("SPY_ret"), pl.col(f"{symbol}_ret"), window_size=63).alias(symbol)
        for symbol in ("QQQ", "IWM", "XLE", "XLF")
    ],
)

figure = go.Figure()
for symbol in ("QQQ", "IWM", "XLE", "XLF"):
    figure.add_scatter(x=rolling_correlations["date"], y=rolling_correlations[symbol], mode="lines", name=f"SPY / {symbol}")

figure.update_layout(
    title="SPY — 63-session rolling return correlation",
    yaxis_title="correlation",
    legend_title_text="",
)
figure.show()


In [ ]:
dispersion_panel = returns_panel.with_columns(
    pl.concat_list([pl.col(column) for column in return_columns]).list.std().alias("cross_sectional_dispersion")
)
dispersion = dispersion_panel.get_column("cross_sectional_dispersion")
dispersion_summary = pl.DataFrame([{
    "mean (%)": dispersion.mean() * 100,
    "median (%)": dispersion.median() * 100,
    "p90 (%)": dispersion.quantile(0.90) * 100,
    "max (%)": dispersion.max() * 100,
}]).with_columns(pl.all().round(3))
dispersion_summary


In [ ]:
figure = go.Figure()
figure.add_scatter(
    x=dispersion_panel["date"],
    y=dispersion_panel["cross_sectional_dispersion"] * 100,
    mode="lines",
    name="cross-sectional dispersion",
)
figure.update_layout(
    title="Daily cross-sectional dispersion — factor proxy returns",
    yaxis_title="standard deviation (%)",
    legend_title_text="",
)
figure.show()


### <b>Interpretación</b>

- QQQ e IWF son el par más correlacionado (0.983); SPY e IWF también se mueven de forma muy próxima (0.958).
- XLE muestra la mayor independencia relativa frente a QQQ (0.391) e IWF (0.423); IWM y XLF conservan co-movimiento relevante con el mercado amplio.
- Los acumulados muestran liderazgo relativo prolongado de growth frente a value: IWF - IWD termina en +40.4%; QQQ - SPY termina en +40.9%.
- Las correlaciones móviles cambian materialmente: SPY/XLE llega a ser negativa en algunos tramos y supera 0.9 en otros.
- La dispersión cross-sectional mediana es 0.574% diario, con p90 de 1.206%; hay heterogeneidad suficiente para estudiar momentum relativo, sin concluir todavía que exista factor momentum.


### <b>Block 02 conclusion</b>

- Se construyeron retornos logarítmicos diarios para los siete proxies sobre 2,231 sesiones comunes.
- La estructura de correlación combina pares muy cercanos, como QQQ/IWF, con comportamiento sectorial más diferenciado, especialmente XLE.
- Los spreads relativos y la dispersión muestran variación transversal, pero no son todavía señales ni resultados de una estrategia.
- No se estimaron betas, regresiones, rankings ni objetos de cartera.

Los retornos quedan preparados para estudiar persistencia temporal y relativa en Block 03.


## <b>Block 03 — Factor momentum</b>

> ¿Los proxies que muestran mejor momentum reciente tienden a mantener liderazgo relativo en horizontes posteriores?

El momentum continuo es $m_t^{(k)} = ln(P_t / P_{t-k})$. Cada observación usa sólo precios conocidos hasta la fecha `t`; los ranks describen posición relativa y no son señales long/short.


In [ ]:
MOMENTUM_HORIZONS = (21, 63, 126, 252)
momentum_panel = common_panel.with_columns([
    (pl.col(symbol).log() - pl.col(symbol).shift(horizon).log()).alias(
        f"{symbol}_mom_{horizon}"
    )
    for symbol in PROXIES
    for horizon in MOMENTUM_HORIZONS
])

rank_panels = {}
for horizon in MOMENTUM_HORIZONS:
    momentum_columns = [f"{symbol}_mom_{horizon}" for symbol in PROXIES]
    valid = momentum_panel.select("date", *momentum_columns).drop_nulls()
    rank_rows = []
    for row in valid.iter_rows(named=True):
        ordered = sorted(PROXIES, key=lambda symbol: row[f"{symbol}_mom_{horizon}"], reverse=True)
        rank_rows.append({
            "date": row["date"],
            **{f"{symbol}_rank": rank for rank, symbol in enumerate(ordered, start=1)},
        })
    rank_panels[horizon] = pl.DataFrame(rank_rows).sort("date")

{horizon: rank_panels[horizon].height for horizon in MOMENTUM_HORIZONS}


### <b>Rank persistence</b>

Spearman compara la ordenación transversal actual con la observada 21 o 63 sesiones después. Un valor positivo indica que la jerarquía relativa tiende a conservarse; no mide rentabilidad.


In [ ]:
spearman_rows = []
rank_variance = sum((rank - 4) ** 2 for rank in range(1, 8))

for horizon, rank_panel in rank_panels.items():
    ranks = {symbol: rank_panel.get_column(f"{symbol}_rank").to_list() for symbol in PROXIES}
    for forward_sessions in (21, 63):
        for index in range(rank_panel.height - forward_sessions):
            spearman = sum(
                (ranks[symbol][index] - 4) * (ranks[symbol][index + forward_sessions] - 4)
                for symbol in PROXIES
            ) / rank_variance
            spearman_rows.append({
                "formation": f"{horizon}d",
                "forward": f"{forward_sessions}d",
                "spearman": spearman,
            })

persistence_observations = pl.DataFrame(spearman_rows)
persistence_summary = (
    persistence_observations
    .group_by("formation", "forward")
    .agg(
        pl.col("spearman").mean().alias("mean"),
        pl.col("spearman").median().alias("median"),
        pl.col("spearman").quantile(0.10).alias("p10"),
        pl.col("spearman").quantile(0.90).alias("p90"),
        (pl.col("spearman") > 0).mean().mul(100).alias("positive (%)"),
    )
    .sort("formation", "forward")
    .with_columns(pl.all().exclude(["formation", "forward"]).round(3))
)
persistence_summary


In [ ]:
extreme_rows = []
for horizon, rank_panel in rank_panels.items():
    ranks = {symbol: rank_panel.get_column(f"{symbol}_rank").to_list() for symbol in PROXIES}
    for forward_sessions in (21, 63):
        leaders_top3 = []
        laggards_bottom3 = []
        for index in range(rank_panel.height - forward_sessions):
            leader = next(symbol for symbol in PROXIES if ranks[symbol][index] == 1)
            laggard = next(symbol for symbol in PROXIES if ranks[symbol][index] == 7)
            leaders_top3.append(ranks[leader][index + forward_sessions] <= 3)
            laggards_bottom3.append(ranks[laggard][index + forward_sessions] >= 5)
        extreme_rows.append({
            "formation": f"{horizon}d",
            "forward": f"{forward_sessions}d",
            "leader remains top 3 (%)": sum(leaders_top3) / len(leaders_top3) * 100,
            "laggard remains bottom 3 (%)": sum(laggards_bottom3) / len(laggards_bottom3) * 100,
        })

extreme_persistence = pl.DataFrame(extreme_rows).with_columns(
    pl.all().exclude(["formation", "forward"]).round(3)
)
extreme_persistence


### <b>126d rank transitions</b>

> La matriz compara grupos de rank actuales con los observados 21 sesiones después: Top 2, Middle 3 y Bottom 2.


In [ ]:
def rank_group(rank):
    if rank <= 2:
        return "Top 2"
    if rank <= 5:
        return "Middle 3"
    return "Bottom 2"

rank_126 = rank_panels[126]
ranks_126 = {symbol: rank_126.get_column(f"{symbol}_rank").to_list() for symbol in PROXIES}
groups = ("Top 2", "Middle 3", "Bottom 2")
transition_counts = {current: {future: 0 for future in groups} for current in groups}

for index in range(rank_126.height - 21):
    for symbol in PROXIES:
        transition_counts[rank_group(ranks_126[symbol][index])][rank_group(ranks_126[symbol][index + 21])] += 1

transition_matrix = pl.DataFrame([
    {
        "current group": current,
        **{future: transition_counts[current][future] / sum(transition_counts[current].values()) * 100 for future in groups},
    }
    for current in groups
]).with_columns(pl.all().exclude("current group").round(3))
transition_matrix


### <b>Mechanical temporal example</b>

La fecha se toma del punto medio de la muestra válida de 126d; no se selecciona por desempeño.


In [ ]:
example_index = rank_126.height // 2
example_dates = {
    "current": rank_126.get_column("date")[example_index],
    "21 sessions later": rank_126.get_column("date")[example_index + 21],
}

def rank_snapshot(label, snapshot_date):
    momentums = momentum_panel.filter(pl.col("date") == snapshot_date).row(named=True)
    ranks = rank_126.filter(pl.col("date") == snapshot_date).row(named=True)
    return pl.DataFrame([
        {
            "snapshot": label,
            "date": snapshot_date,
            "symbol": symbol,
            "momentum_126 (%)": momentums[f"{symbol}_mom_126"] * 100,
            "rank": ranks[f"{symbol}_rank"],
        }
        for symbol in PROXIES
    ]).sort("rank").with_columns(pl.col("momentum_126 (%)").round(3))

example_now = rank_snapshot("current", example_dates["current"])
example_now


In [ ]:
example_forward = rank_snapshot("21 sessions later", example_dates["21 sessions later"])
example_forward


In [ ]:
figure = go.Figure()
for symbol in ("QQQ", "IWD", "XLE", "IWM"):
    figure.add_scatter(x=rank_126["date"], y=rank_126[f"{symbol}_rank"], mode="lines", name=symbol)

figure.update_layout(
    title="126d momentum rank — selected factor proxies",
    yaxis_title="rank (1 = highest momentum)",
    yaxis_autorange="reversed",
    legend_title_text="",
)
figure.show()


In [ ]:
leader_rows = []
for symbol in PROXIES:
    ranks = rank_126.get_column(f"{symbol}_rank")
    leader_rows.append({
        "symbol": symbol,
        "times_rank_1": int((ranks == 1).sum()),
        "pct_rank_1": (ranks == 1).mean() * 100,
        "times_top3": int((ranks <= 3).sum()),
        "pct_top3": (ranks <= 3).mean() * 100,
    })

leader_frequency = pl.DataFrame(leader_rows).with_columns(
    pl.col(["pct_rank_1", "pct_top3"]).round(3)
).sort("times_rank_1", descending=True)
qqq_ranks = rank_126.get_column("QQQ_rank").to_list()
iwf_ranks = rank_126.get_column("IWF_rank").to_list()
qqq_mean = sum(qqq_ranks) / len(qqq_ranks)
iwf_mean = sum(iwf_ranks) / len(iwf_ranks)
rank_correlation = sum(
    (qqq - qqq_mean) * (iwf - iwf_mean) for qqq, iwf in zip(qqq_ranks, iwf_ranks)
) / ((sum((qqq - qqq_mean) ** 2 for qqq in qqq_ranks) * sum((iwf - iwf_mean) ** 2 for iwf in iwf_ranks)) ** 0.5)
rank_alignment = pl.DataFrame([{
    "QQQ / IWF 126d rank correlation": rank_correlation,
    "both in top 3 (%)": ((rank_126["QQQ_rank"] <= 3) & (rank_126["IWF_rank"] <= 3)).mean() * 100,
}]).with_columns(pl.all().round(3))
leader_frequency


### <b>Interpretación</b>

- Los ranks de 21d prácticamente no persisten: Spearman medio es 0.023 a 21 sesiones y 0.001 a 63 sesiones.
- La persistencia aumenta con 126d y 252d; sin embargo, sus ventanas de formación se solapan con los horizontes posteriores, por lo que parte de esa continuidad es mecánica y no evidencia predictiva independiente.
- En 126d, 87.1% de los líderes permanece en top 3 y 91.7% de los perdedores en bottom 3 tras 21 sesiones; la matriz también conserva 70.9% de Top 2 y 74.1% de Bottom 2.
- QQQ aparece primero con mayor frecuencia en 126d (30.1%), pero XLE también lidera 27.2% de las fechas, lo que muestra rotación sectorial además de liderazgo growth.
- QQQ e IWF conservan redundancia parcial en ranking (correlación 0.675 y ambos en top 3 el 51.1% de las fechas); XLE aporta episodios de liderazgo distintos.


### <b>Block 03 conclusion</b>

- Se calcularon medidas continuas de momentum de 21d, 63d, 126d y 252d para cada proxy, siempre con información disponible hasta `t`.
- El bloque separa la persistencia temporal de cada medida de momentum de la persistencia relativa de su ranking transversal.
- Los ranks largos conservan grupos de liderazgo con fuerza, mientras que los ranks 21d rotan casi por completo.
- El solapamiento entre lookback y horizonte futuro limita una lectura predictiva de la persistencia alta en 126d y 252d.
- No se construyeron señales, carteras ni resultados de rentabilidad.

El siguiente bloque estudiará si el momentum observado en un activo puede relacionarse con el momentum de los proxies sistemáticos.


## <b>Block 03B — Forward persistence</b>

> Este bloque separa la continuidad mecánica de los ranks de la relación entre el momentum conocido en $t$ y los retornos realizados después de $t$.

Para cada proxy, el retorno futuro es $fwd_{t,h} = \ln(P_{t+h}/P_t)$. La ventana empieza después de $t$: usa los retornos de $t+1$ a $t+h$ y no reutiliza retornos de la ventana de formación.

In [ ]:
FORMATION_HORIZONS = (63, 126, 252)
FORWARD_HORIZONS = (21, 63, 126)

forward_panel = common_panel.with_columns([
    (pl.col(symbol).shift(-horizon).log() - pl.col(symbol).log()).alias(
        f"{symbol}_fwd_{horizon}"
    )
    for symbol in PROXIES
    for horizon in FORWARD_HORIZONS
])

forward_panel.select("date", "SPY_fwd_21", "SPY_fwd_63", "SPY_fwd_126").tail(3)

### <b>Current rank and future return</b>

El rank de formación usa 1 para el mayor momentum. Para Spearman, los retornos futuros también se rankean con 1 como el mayor retorno realizado; así, una correlación positiva significa que un mejor rank actual coincide con un mejor retorno posterior.

In [ ]:
forward_rows = []

for formation in FORMATION_HORIZONS:
    rank_panel = rank_panels[formation]
    for forward in FORWARD_HORIZONS:
        future_columns = [f"{symbol}_fwd_{forward}" for symbol in PROXIES]
        paired = rank_panel.join(
            forward_panel.select("date", *future_columns).drop_nulls(),
            on="date",
            how="inner",
        ).sort("date")

        for row in paired.iter_rows(named=True):
            current_rank = {symbol: row[f"{symbol}_rank"] for symbol in PROXIES}
            future_return = {symbol: row[f"{symbol}_fwd_{forward}"] for symbol in PROXIES}
            future_order = sorted(PROXIES, key=lambda symbol: future_return[symbol], reverse=True)
            future_rank = {symbol: rank for rank, symbol in enumerate(future_order, start=1)}

            spearman = sum(
                (current_rank[symbol] - 4) * (future_rank[symbol] - 4)
                for symbol in PROXIES
            ) / 28
            top2 = [symbol for symbol in PROXIES if current_rank[symbol] <= 2]
            bottom2 = [symbol for symbol in PROXIES if current_rank[symbol] >= 6]
            leader = next(symbol for symbol in PROXIES if current_rank[symbol] == 1)
            laggard = next(symbol for symbol in PROXIES if current_rank[symbol] == 7)
            top2_return = sum(future_return[symbol] for symbol in top2) / 2
            bottom2_return = sum(future_return[symbol] for symbol in bottom2) / 2

            forward_rows.append({
                "date": row["date"],
                "formation_sessions": formation,
                "forward_sessions": forward,
                "spearman": spearman,
                "top2_forward_return": top2_return,
                "bottom2_forward_return": bottom2_return,
                "top_minus_bottom": top2_return - bottom2_return,
                "leader_forward_return": future_return[leader],
                "cross_sectional_mean": sum(future_return.values()) / len(PROXIES),
                "rank7_forward_return": future_return[laggard],
            })

forward_observations = pl.DataFrame(forward_rows)
forward_observations.head(3)

In [ ]:
forward_summary = (
    forward_observations
    .group_by("formation_sessions", "forward_sessions")
    .agg(
        pl.col("spearman").mean().alias("mean_spearman"),
        pl.col("spearman").median().alias("median_spearman"),
        (pl.col("spearman") > 0).mean().mul(100).alias("positive_pct"),
    )
    .sort("formation_sessions", "forward_sessions")
    .with_columns(pl.all().exclude("formation_sessions", "forward_sessions").round(3))
)
forward_summary

### <b>Top 2 versus Bottom 2</b>

En cada fecha se comparan los dos mayores y los dos menores ranks de formación con sus retornos logarítmicos futuros. El spread es $\overline{fwd}_{Top\ 2} - \overline{fwd}_{Bottom\ 2}$ y se mantiene en su horizonte realizado, sin anualizar.

In [ ]:
spread_summary = (
    forward_observations
    .group_by("formation_sessions", "forward_sessions")
    .agg(
        pl.col("top2_forward_return").mean().mul(100).alias("top2_mean_pct"),
        pl.col("bottom2_forward_return").mean().mul(100).alias("bottom2_mean_pct"),
        pl.col("top_minus_bottom").mean().mul(100).alias("mean_spread_pct"),
        pl.col("top_minus_bottom").median().mul(100).alias("median_spread_pct"),
        (pl.col("top_minus_bottom") > 0).mean().mul(100).alias("spread_positive_pct"),
    )
    .sort("formation_sessions", "forward_sessions")
    .with_columns(pl.all().exclude("formation_sessions", "forward_sessions").round(3))
)
spread_summary

### <b>Forward-window check</b>

Las combinaciones 126d/21d y 126d/126d usan formación hasta $t$ y evaluación desde $t+1$, por lo que no comparten retornos diarios con la ventana de formación. Las observaciones evaluadas en fechas consecutivas sí pueden compartir días entre sus propias ventanas futuras; esa dependencia se revisa después con una muestra cada 21 sesiones.

In [ ]:
forward_summary.filter(
    (pl.col("formation_sessions") == 126)
    & pl.col("forward_sessions").is_in([21, 126])
)

### <b>Evaluation-frequency sensitivity</b>

La comparación 126d/21d se resume tanto diariamente como tomando una fecha de evaluación cada 21 sesiones. La segunda vista reduce el solapamiento entre ventanas futuras consecutivas.

In [ ]:
daily_126_21 = forward_observations.filter(
    (pl.col("formation_sessions") == 126) & (pl.col("forward_sessions") == 21)
)
every_21_126_21 = daily_126_21.with_row_index("evaluation_index").filter(
    pl.col("evaluation_index") % 21 == 0
)

sensitivity_rows = []
for label, sample in (("daily", daily_126_21), ("every 21 sessions", every_21_126_21)):
    sensitivity_rows.append({
        "evaluation": label,
        "observations": sample.height,
        "mean_spearman": sample.get_column("spearman").mean(),
        "mean_spread_pct": sample.get_column("top_minus_bottom").mean() * 100,
        "spread_positive_pct": (sample.get_column("top_minus_bottom") > 0).mean() * 100,
    })

sensitivity_summary = pl.DataFrame(sensitivity_rows).with_columns(
    pl.all().exclude("evaluation", "observations").round(3)
)
sensitivity_summary

### <b>Leader outcomes</b>

Para formación de 126d, la tabla compara el retorno futuro del líder actual con la media transversal y con el proxy que ocupaba el último rank. Es una descripción de resultados posteriores, no una cartera ni una prueba de rentabilidad.

In [ ]:
leader_outcomes = (
    forward_observations
    .filter((pl.col("formation_sessions") == 126) & pl.col("forward_sessions").is_in([21, 63]))
    .group_by("forward_sessions")
    .agg(
        pl.col("leader_forward_return").mean().mul(100).alias("leader_mean_pct"),
        pl.col("cross_sectional_mean").mean().mul(100).alias("cross_sectional_mean_pct"),
        pl.col("rank7_forward_return").mean().mul(100).alias("rank7_mean_pct"),
    )
    .sort("forward_sessions")
    .with_columns(pl.all().exclude("forward_sessions").round(3))
)
leader_outcomes

In [ ]:
spread_126_21 = forward_observations.filter(
    (pl.col("formation_sessions") == 126) & (pl.col("forward_sessions") == 21)
)

figure = go.Figure()
figure.add_scatter(
    x=spread_126_21["date"],
    y=spread_126_21["top_minus_bottom"] * 100,
    mode="lines",
    name="Top 2 − Bottom 2",
)
figure.add_hline(y=0, line_dash="dash", line_color="gray")
figure.update_layout(
    title="126d formation / 21d forward — Top 2 minus Bottom 2",
    yaxis_title="forward log-return spread (%)",
    xaxis_title="formation date",
    template="plotly_white",
)
figure.show()

### <b>Interpretación</b>

- La relación forward es positiva pero moderada: Spearman medio va de 0.030 en 63d/21d a 0.155 en 63d/126d; los horizontes realizados más largos concentran las lecturas más altas.
- Para 126d/21d, el spread medio Top 2 − Bottom 2 es 0.274% y resulta positivo el 53.9% de las veces, una diferencia pequeña y ruidosa.
- El resumen diario de 126d/21d (Spearman 0.047; spread 0.274%) cambia poco al evaluar cada 21 sesiones (0.072; 0.239%), aunque esta muestra tiene menos observaciones.
- El líder 126d supera en promedio a la media transversal y al último rank a 21d y 63d, pero los resultados siguen expuestos a concentración entre QQQ/IWF y a episodios diferenciados de XLE.
- Los spreads describen resultados posteriores entre grupos extremos; no representan una cartera implementable ni incluyen costes.

### <b>Block 03B conclusion</b>

- El bloque evalúa retornos estrictamente posteriores a cada fecha de formación y los separa de la persistencia mecánica de ranks de Block 03.
- La evidencia forward es positiva pero modesta; los horizontes realizados de 126 sesiones muestran los Spearman y spreads medios más altos.
- La comparación cada 21 sesiones conserva una señal similar para 126d/21d, aunque con una muestra mucho menor.
- La ventaja media del líder frente al último rank existe en esta muestra, pero varía por proxy y no equivale a una estimación de alpha.
- No se construyeron carteras, señales operables ni resultados netos de costes.

La evidencia descriptiva es suficiente para que el siguiente bloque examine cómo el momentum de un activo se relaciona con el de estos proxies sistemáticos.

## <b>Block 04 — Asset vs Factor Momentum</b>

> Este bloque separa el momentum total de QQQ, XLE y XLF entre una parte asociada a proxies sistemáticos observables y una parte residual.

El modelo diario es $R_{i,t} = alpha_i + beta_{mkt}MKT_t + beta_{size}SIZE_t + beta_{gv}GV_t + residual_{i,t}$. Los componentes son proxies ETF construidos, no factores académicos puros.

In [ ]:
import numpy as np

TARGETS = ("QQQ", "XLE", "XLF")
FACTOR_COLUMNS = ("market", "size", "growth_value")
ROLLING_WINDOW = 252

model_panel = (
    returns_panel
    .select("date", *[f"{asset}_ret" for asset in TARGETS], "SPY_ret", "IWM_ret", "IWF_ret", "IWD_ret")
    .with_columns(
        pl.col("SPY_ret").alias("market"),
        (pl.col("IWM_ret") - pl.col("SPY_ret")).alias("size"),
        (pl.col("IWF_ret") - pl.col("IWD_ret")).alias("growth_value"),
    )
    .select("date", *[f"{asset}_ret" for asset in TARGETS], *FACTOR_COLUMNS)
    .drop_nulls()
)

factor_values = model_panel.select(*FACTOR_COLUMNS).to_numpy()
standardized_factors = (factor_values - factor_values.mean(axis=0)) / factor_values.std(axis=0, ddof=1)
factor_correlation = np.corrcoef(standardized_factors, rowvar=False)
factor_pairs = [(abs(factor_correlation[i, j]), FACTOR_COLUMNS[i], FACTOR_COLUMNS[j]) for i in range(3) for j in range(i + 1, 3)]
max_correlation, factor_a, factor_b = max(factor_pairs)

factor_diagnostics = pl.DataFrame([{
    "observations": model_panel.height,
    "largest_abs_factor_corr": max_correlation,
    "pair": f"{factor_a} / {factor_b}",
    "standardized_condition_number": np.linalg.cond(standardized_factors),
}]).with_columns(pl.all().exclude("observations", "pair").round(3))
factor_diagnostics

### <b>Rolling alignment</b>

En cada fecha $t$, OLS usa las 252 sesiones disponibles hasta $t$. Esas betas se aplican a los proxies observados en $t+1$ para obtener $R_{systematic,t+1}$; por tanto, el retorno objetivo de $t+1$ no participa en la estimación que lo explica.

In [ ]:
dates = model_panel.get_column("date").to_list()
factors = model_panel.select(*FACTOR_COLUMNS).to_numpy()
rolling_rows = []

for asset in TARGETS:
    target_returns = model_panel.get_column(f"{asset}_ret").to_numpy()
    for end in range(ROLLING_WINDOW - 1, len(target_returns) - 1):
        window_factors = factors[end - ROLLING_WINDOW + 1:end + 1]
        design = np.column_stack((np.ones(ROLLING_WINDOW), window_factors))
        coefficients = np.linalg.lstsq(design, target_returns[end - ROLLING_WINDOW + 1:end + 1], rcond=None)[0]
        fitted = design @ coefficients
        actual_window = target_returns[end - ROLLING_WINDOW + 1:end + 1]
        rolling_r2 = 1 - ((actual_window - fitted) ** 2).sum() / ((actual_window - actual_window.mean()) ** 2).sum()

        next_systematic = coefficients[0] + factors[end + 1] @ coefficients[1:]
        next_actual = target_returns[end + 1]
        rolling_rows.append({
            "date": dates[end + 1],
            "beta_asof_date": dates[end],
            "asset": asset,
            "alpha": coefficients[0],
            "beta_market": coefficients[1],
            "beta_size": coefficients[2],
            "beta_growth_value": coefficients[3],
            "rolling_r2": rolling_r2,
            "actual_return": next_actual,
            "systematic_return": next_systematic,
            "residual_return": next_actual - next_systematic,
        })

rolling_models = pl.DataFrame(rolling_rows)
rolling_models.select("date", "beta_asof_date", "asset", "actual_return", "systematic_return", "residual_return").head(3)

In [ ]:
beta_rows = []
for asset in TARGETS:
    for beta, label in (("beta_market", "market"), ("beta_size", "size"), ("beta_growth_value", "growth/value")):
        values = rolling_models.filter(pl.col("asset") == asset).get_column(beta)
        beta_rows.append({
            "asset": asset,
            "proxy": label,
            "mean": values.mean(),
            "p10": values.quantile(0.10),
            "median": values.median(),
            "p90": values.quantile(0.90),
        })

beta_summary = pl.DataFrame(beta_rows).with_columns(pl.all().exclude("asset", "proxy").round(3))
beta_summary

In [ ]:
qqq_betas = rolling_models.filter(pl.col("asset") == "QQQ")

figure = go.Figure()
for column, label in (("beta_market", "market"), ("beta_size", "size"), ("beta_growth_value", "growth/value")):
    figure.add_scatter(x=qqq_betas["date"], y=qqq_betas[column], mode="lines", name=label)

figure.add_hline(y=0, line_dash="dash", line_color="gray")
figure.update_layout(
    title="QQQ — rolling systematic-proxy exposures",
    xaxis_title="prediction date",
    yaxis_title="rolling beta",
    template="plotly_white",
)
figure.show()

In [ ]:
reconstruction_audit = (
    rolling_models
    .with_columns((pl.col("actual_return") - pl.col("systematic_return") - pl.col("residual_return")).abs().alias("reconstruction_error"))
    .group_by("asset")
    .agg(pl.col("reconstruction_error").max().alias("max_abs_reconstruction_error"))
)
r2_summary = (
    rolling_models
    .group_by("asset")
    .agg(
        pl.col("rolling_r2").mean().alias("mean_rolling_r2"),
        pl.col("rolling_r2").median().alias("median_rolling_r2"),
        pl.col("rolling_r2").quantile(0.10).alias("r2_p10"),
        pl.col("rolling_r2").quantile(0.90).alias("r2_p90"),
    )
)
model_audit = reconstruction_audit.join(r2_summary, on="asset").with_columns(pl.all().exclude("asset").round(4))
model_audit

### <b>Total, systematic and residual momentum</b>

Cada componente acumula retornos logarítmicos causales sobre 63, 126 y 252 sesiones. Por construcción, $MOM_{total}$ es la suma de $MOM_{systematic}$ y $MOM_{residual}$ en cada ventana.

In [ ]:
COMPONENT_HORIZONS = (63, 126, 252)
component_rows = []

for asset in TARGETS:
    asset_frame = rolling_models.filter(pl.col("asset") == asset).sort("date")
    component_dates = asset_frame.get_column("date").to_list()
    actual = asset_frame.get_column("actual_return").to_numpy()
    systematic = asset_frame.get_column("systematic_return").to_numpy()
    residual = asset_frame.get_column("residual_return").to_numpy()

    for horizon in COMPONENT_HORIZONS:
        for end in range(horizon - 1, len(asset_frame)):
            start = end - horizon + 1
            component_rows.append({
                "date": component_dates[end],
                "asset": asset,
                "horizon_sessions": horizon,
                "total_momentum": actual[start:end + 1].sum(),
                "systematic_momentum": systematic[start:end + 1].sum(),
                "residual_momentum": residual[start:end + 1].sum(),
            })

momentum_components = pl.DataFrame(component_rows)
momentum_components.head(3)

In [ ]:
momentum_relationship = (
    momentum_components
    .group_by("asset", "horizon_sessions")
    .agg(
        pl.corr("total_momentum", "systematic_momentum").alias("total_systematic_corr"),
        pl.corr("total_momentum", "residual_momentum").alias("total_residual_corr"),
    )
    .sort("asset", "horizon_sessions")
    .with_columns(pl.all().exclude("asset", "horizon_sessions").round(3))
)
momentum_relationship

In [ ]:
qqq_momentum_126 = momentum_components.filter(
    (pl.col("asset") == "QQQ") & (pl.col("horizon_sessions") == 126)
)

figure = go.Figure()
for column, label in (("total_momentum", "total"), ("systematic_momentum", "systematic"), ("residual_momentum", "residual")):
    figure.add_scatter(x=qqq_momentum_126["date"], y=qqq_momentum_126[column] * 100, mode="lines", name=label)

figure.add_hline(y=0, line_dash="dash", line_color="gray")
figure.update_layout(
    title="QQQ — 126d total, systematic and residual momentum",
    xaxis_title="date",
    yaxis_title="cumulative log return (%)",
    template="plotly_white",
)
figure.show()

In [ ]:
qqq_divergences = (
    qqq_momentum_126
    .filter(
        ((pl.col("total_momentum") > 0) & (pl.col("residual_momentum") < 0))
        | ((pl.col("total_momentum") < 0) & (pl.col("residual_momentum") > 0))
    )
    .with_columns(
        pl.when((pl.col("total_momentum") > 0) & (pl.col("residual_momentum") < 0))
        .then(pl.lit("total positive / residual negative"))
        .otherwise(pl.lit("total negative / residual positive"))
        .alias("case"),
    )
    .group_by("case", maintain_order=True)
    .head(4)
    .select("date", "case", "total_momentum", "systematic_momentum", "residual_momentum")
    .with_columns(pl.all().exclude("date", "case").mul(100).round(3))
    .sort("date")
)
qqq_divergences

### <b>Forward information</b>

Como revisión descriptiva, se compara cada momentum de 126d conocido en $t$ con el retorno realizado del activo durante las 21 sesiones posteriores. Spearman positivo indica que valores mayores del componente coinciden, en promedio, con mayor retorno futuro.

In [ ]:
forward_information_rows = []
for asset in TARGETS:
    component_frame = momentum_components.filter(
        (pl.col("asset") == asset) & (pl.col("horizon_sessions") == 126)
    )
    paired = component_frame.join(
        forward_panel.select("date", f"{asset}_fwd_21").drop_nulls(),
        on="date",
        how="inner",
    )
    future_rank = pl.Series(paired.get_column(f"{asset}_fwd_21")).rank("average").to_numpy()
    for column, label in (("total_momentum", "total"), ("systematic_momentum", "systematic"), ("residual_momentum", "residual")):
        component_rank = pl.Series(paired.get_column(column)).rank("average").to_numpy()
        spearman = np.corrcoef(component_rank, future_rank)[0, 1]
        forward_information_rows.append({
            "asset": asset,
            "component": label,
            "spearman_126d_to_21d": spearman,
            "sign": "positive" if spearman > 0 else "negative",
            "abs_spearman": abs(spearman),
        })

forward_information = pl.DataFrame(forward_information_rows).with_columns(
    pl.all().exclude("asset", "component", "sign").round(3)
)
forward_information

### <b>Interpretación</b>

- QQQ queda fuertemente asociado a mercado y growth/value: sus betas medias son 1.031 y 0.596, respectivamente, y su R2 rolling mediano es 0.980.
- XLE combina beta de mercado media de 1.091 con growth/value negativo de -0.958; su R2 mediano de 0.508 deja una fracción residual materialmente mayor.
- XLF mantiene beta de mercado elevada (1.144) y growth/value negativo (-0.674), con R2 mediano de 0.844; sus exposiciones difieren de las de QQQ.
- En momentum 126d, la correlación total-sistemático es 0.981 para QQQ, 0.835 para XLE y 0.901 para XLF; la componente residual es especialmente distinta en XLE.
- La mayor magnitud forward en esta revisión es el residual de XLE (-0.126 Spearman); es descriptiva y no identifica alpha residual ni una regla de trading.

### <b>Block 04 conclusion</b>

- QQQ parece dominado por proxies de mercado y growth/value, mientras que XLE conserva una componente residual comparativamente mayor; XLF ocupa una posición intermedia.
- Las exposiciones se reestiman en el tiempo y la baja colinealidad entre los tres proxies evita una inestabilidad obvia de coeficientes en esta especificación.
- El momentum total mantiene alta relación con el sistemático, especialmente en QQQ, pero los componentes residuales muestran variación distinta, sobre todo en XLE.
- Las correlaciones forward son pequeñas y mixtas; no constituyen evidencia de alpha residual ni de una estrategia implementable.
- La identidad entre retorno total, sistemático y residual se cumple a precisión numérica.

El siguiente paso lógico será profundizar en la estabilidad y la interpretación económica de los componentes antes de considerar cualquier construcción de cartera.

## <b>Block 05 — Diagnostics</b>

> Revisar cobertura, concentración, correlaciones y límites de los proxies antes de profundizar en la relación entre activos y factores.

_Placeholder — sin cálculos todavía._
